In [1]:
from pathlib import Path
import numpy as np
import os, json, random, pickle
from collections import Counter
from pathlib import Path

In [2]:
# Check image path

omama_dir_path = "/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/2d_resized_256/images"
if not os.path.exists(omama_dir_path):
    print(f"File not found: {omama_dir_path}")
    print("Please update dicom_path with a valid image file path")
else:
    print(f"Found DICOM folder: {os.path.basename(omama_dir_path)}")
    omama_folder = Path("/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/2d_resized_256/images")
    IDs = sorted(str(file) for file in omama_folder.rglob("*.npz"))
    print(len(IDs))

Found DICOM folder: images
163568


In [3]:
print(Path(IDs[0]).stem)

100000039159562031368112550794429920461


In [4]:
meta_dir = Path("/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/2d_resized_256/metadata")

In [5]:
# Get the headers
pkl = Path("/home/anya.tongprasith001/U54REC/release_to_header_mapping.pkl")
mapping = pickle.load(open(pkl, "rb"))
len(mapping)

163568

In [6]:
# Get the images, labels, and headers
noncancer = {}
cancer = {}

for ID in IDs :
    meta = json.load(open(meta_dir / f"{Path(ID).stem}.json"))
    if meta["label"] == "Unknown" :
        continue
        
    # Load image
    img_name = ID
    # Load headers
    ds = mapping[Path(ID).stem]
    bdthickness = ds.BodyPartThickness
    age_str = ds.PatientAge
    if age_str and age_str != '':
        age = float(age_str[:-1]) / 12.0
    else:
        age = -1.0 
            
    if meta["label"] == "NonCancer" :
        if meta["PatientID"] not in noncancer.keys() :
            noncancer[meta["PatientID"]] = []
        noncancer[meta["PatientID"]].append([img_name, [bdthickness, age], 0])

    else : 
        if meta["PatientID"] not in cancer.keys() :
            cancer[meta["PatientID"]] = []
        cancer[meta["PatientID"]].append([img_name, [bdthickness, age], 1])

In [7]:
noncancer_ls = list(noncancer.items())
c = list(cancer.items())

print(len(noncancer_ls), len(c))

154238 3562


In [8]:
count = 0
nc_groups = []
for i in range(10) :
    nc = noncancer_ls[count:count + len(cancer.keys())]
    count += len(cancer.keys())
    nc_groups.append(nc)

In [9]:
print(len(nc_groups[0]))

3562


In [10]:
# Split patients and shuffle them
train_size = int(0.7 * len(nc_groups[0])) 
val_size = int(0.15 * len(nc_groups[0]))
test_size = int(0.15 * len(nc_groups[0]))

train_ncp, train_cp = nc_groups[0][:train_size], c[:train_size]
val_ncp, val_cp = nc_groups[0][train_size:val_size + train_size], c[train_size:val_size + train_size]
test_ncp, test_cp = nc_groups[0][val_size + train_size:val_size + train_size + test_size], c[val_size + train_size:val_size + train_size + test_size]

print(f"Noncancer Train: {len(train_ncp)}, Val: {len(val_ncp)}, Test: {len(test_ncp)}")
print(f"Cancer Train: {len(train_ncp)}, Val: {len(val_ncp)}, Test: {len(test_ncp)}")

train_ds = train_ncp + train_cp
val_ds = val_ncp + val_cp
test_ds = test_ncp + test_cp
random.shuffle(train_ds)
random.shuffle(val_ds)
random.shuffle(test_ds)


Noncancer Train: 2493, Val: 534, Test: 534
Cancer Train: 2493, Val: 534, Test: 534


In [ ]:
# Assign the images, metadata, and labels for setting up mix input
def assign_data(dataset) :
    images, metadata, labels = [],[],[]
    for patient, data_list in dataset :
        for data in data_list :
            img = np.load(data[0])['data']
            img = np.expand_dims(img, axis=-1)
            images.append(img)
            metadata.append(data[1])
            labels.append(data[2])
    return images, metadata, labels

train_imgs, train_metadata, train_labels = assign_data(train_ds)
val_imgs, val_metadata, val_labels = assign_data(val_ds)
test_imgs, test_metadata, test_labels = assign_data(test_ds)

train_imgs = np.array(train_imgs, dtype=np.float32)
train_metadata = np.array(train_metadata, dtype=np.float32)
train_labels = np.array(train_labels, dtype=np.float32)
val_imgs = np.array(val_imgs, dtype=np.float32)
val_metadata = np.array(val_metadata, dtype=np.float32)
val_labels = np.array(val_labels, dtype=np.float32)
test_imgs = np.array(test_imgs, dtype=np.float32)
test_metadata = np.array(test_metadata, dtype=np.float32)
test_labels = np.array(test_labels, dtype=np.float32)

In [ ]:
# Check the distribution of classes
print(f'Average class probability in training set:   {train_labels.mean():.4f}')
print(f'Average class probability in validation set: {val_labels.mean():.4f}')
print(f'Average class probability in test set:       {test_labels.mean():.4f}')

In [ ]:
# Save this training ds
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/train1_imgs.npy', train_imgs)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/train1_metadata.npy', train_metadata)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/train1_labels.npy', train_labels)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/val1_imgs.npy', val_imgs)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/val1_metadata.npy', val_metadata)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/val1_labels.npy', val_labels)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/test1_imgs.npy', test_imgs)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/test1_metadata.npy', test_metadata)
np.save('/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/split/test1_labels.npy', test_labels)